# Test Fine-tuning Results
Test the fine-tuned LIANet Creoss region performance to the local model perfomance

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add src to path
sys.path.insert(0, '/home/user/src')
from datasets import PASTIS

from settings import *

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
import json
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

# pretrained_model_path = "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01"

pretrained_model_path_list = {
    "T30UXV": "/home/user/results_shared/fourier_learned_T30UXV/2026-04-06_04-50-55",
    "T31TFJ": "/home/user/results_shared/fourier_learned_T31TFJ/2026-04-02_13-40-05",
    "T31TFM": "/home/user/results_shared/fourier_learned_T31TFM/2026-04-08_00-13-56",
    "T32ULU": "/home/user/results_shared/fourier_learned_T32ULU/2026-04-04_09-21-54"
}

def load_model(CKPT_PATH, other_task):

    pretrained_model_path = pretrained_model_path_list[other_task.split("_")[-1]]
    model_finetune = DownstreamModel(
        model_path=pretrained_model_path,
        checkpoint_path_relative="model_checkpoints/latest_validation_checkpoint.pt",
        adaption_strategy="replace_final_block",
        num_classes=num_classes[other_task],
        activation="none"
    )

    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)
    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    # print("✓ Model loaded successfully")
    return model_finetune

In [3]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import multiclass_segmentation_metrics


all_tasks_list = {
    "PASTIS_local_T31TFM",
    "PASTIS_local_T31TFJ",
    "PASTIS_local_T32ULU",
    "PASTIS_local_T30UXV",
}

VAL_FOLDS = [1, 2, 3, 4, 5]
BATCH_SIZE = 16
NUM_WORKERS = 8

results_rows = []


for Target_region in all_tasks_list:
    source_region = Target_region

    for val_fold in VAL_FOLDS:
        val_dataset = PASTIS(
            top_dir=TOP_DIR[Target_region],
            s2_tiles=s2_tiles[Target_region],
            labels=labels[Target_region],
            train_val_key="val",
            val_folds=[val_fold],
        )

        dataloader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            drop_last=False,
        )

        model_dir = (
            f"/home/user/results_local/finetuning_results/"
            f"{source_region}/LIANet_valFolds{val_fold}_lr0.0001_batchsize16"
        )

        if not os.path.exists(model_dir):
            print(f"Missing folder: {model_dir}")
            continue

        for run_name in sorted(os.listdir(model_dir)):
            ckpt_path = os.path.join(model_dir, run_name, "last.pt")

            if not os.path.exists(ckpt_path):
                continue

            model = load_model(ckpt_path, source_region)
            model.eval()

            list_of_metrics, _ = multiclass_segmentation_metrics(
                num_classes=num_classes[Target_region],
                ignore_index=255,
            )

            metrictracker = MetricCollection(list_of_metrics).to(device)

            with torch.no_grad():
                for batch in tqdm(
                    dataloader,
                    desc=f"{Target_region} | fold {val_fold} | {run_name}",
                    leave=False,
                ):
                    x = batch["x_s2"].to(device)
                    y = batch["y_s2"].to(device)
                    label = batch["label"].to(device)
                    delta_days = batch["delta_days"].to(device)

                    _, pred = model(
                        delta_days,
                        x,
                        y,
                        torch.tensor([0], device=device),
                    )

                    pred = getattr(pred, "output", pred)

                    if pred.dim() == 3:
                        pred = pred.unsqueeze(1)

                    metrictracker.update(pred, label)

            results = metrictracker.compute()

            row = {
                "target_region": Target_region,
                "val_fold": val_fold,
                "source_region": source_region,
                "seed_or_run": run_name,
                "checkpoint_path": ckpt_path,
            }

            for metric_name, metric_value in results.items():
                row[metric_name] = float(metric_value.detach().cpu())

            results_rows.append(row)

            del model
            del metrictracker
            torch.cuda.empty_cache()

        del dataloader
        del val_dataset
        torch.cuda.empty_cache()


df_local_results = pd.DataFrame(results_rows)

df_local_results.to_csv(
    "local_baseline_results_all_regions.csv",
    index=False,
)

df_local_results

Building val image label pairs: 100%|██████████| 19/19 [00:37<00:00,  1.98s/it]


Found 1919 samples for val


Building val image label pairs: 100%|██████████| 19/19 [00:35<00:00,  1.88s/it]                      


Found 2413 samples for val


Building val image label pairs: 100%|██████████| 19/19 [00:35<00:00,  1.87s/it]                      


...

Building val image label pairs: 100%|██████████| 18/18 [00:50<00:00,  2.79s/it]                      


Found 2358 samples for val


Building val image label pairs: 100%|██████████| 18/18 [00:48<00:00,  2.69s/it]                      


Found 2520 samples for val


,target_region,val_fold,source_region,seed_or_run,checkpoint_path,accuracy_macro,accuracy_micro,f1_macro,f1_micro,jaccard_macro,jaccard_micro,precision_macro,precision_micro,recall_macro,recall_micro
0,PASTIS_local_T32ULU,1,PASTIS_local_T32ULU,2026-05-08_12-42-38,/home/user/results_local/finetuning_results/PA...,0.626245,0.748802,0.486114,0.748802,0.361772,0.598469,0.430337,0.748802,0.626245,0.748802
1,PASTIS_local_T32ULU,1,PASTIS_local_T32ULU,2026-05-08_13-27-11,/home/user/results_local/finetuning_results/PA...,0.631234,0.754677,0.491019,0.754677,0.366586,0.606010,0.435092,0.754677,0.631234,0.754677
2,PASTIS_local_T32ULU,1,PASTIS_local_T32ULU,2026-05-08_14-13-07,/home/user/results_local/finetuning_results/PA...,0.621763,0.752012,0.486138,0.752012,0.362452,0.602580,0.430474,0.752012,0.621763,0.752012
3,PASTIS_local_T32ULU,1,PASTIS_local_T32ULU,2026-05-08_14-58-07,/home/user/results_local/finetuning_results/PA...,0.626543,0.750658,0.485107,0.750658,0.360969,0.600842,0.430022,0.750658,0.626543,0.750658
4,PASTIS_local_T32ULU,1,PASTIS_local_T32ULU,2026-05-08_15-43-36,/home/user/results_local/finetuning_results/PA...,0.617986,0.752566,0.484731,0.752566,0.362139,0.603291,0.429940,0.752566,0.617986,0.752566
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,PASTIS_local_T31TFM,5,PASTIS_local_T31TFM,2026-05-09_06-59-47,/home/user/results_local/finetuning_results/PA...,0.548081,0.749085,0.446676,0.749085,0.336779,0.598829,0.408522,0.749085,0.548081,0.749085
96,PASTIS_local_T31TFM,5,PASTIS_local_T31TFM,2026-05-09_07-54-32,/home/user/results_local/finetuning_results/PA...,0.547011,0.751431,0.445239,0.751431,0.336787,0.601834,0.410242,0.751431,0.547011,0.751431
97,PASTIS_local_T31TFM,5,PASTIS_local_T31TFM,2026-05-09_08-49-23,/home/user/results_local/finetuning_results/PA...,0.546414,0.745026,0.439533,0.745026,0.330584,0.593659,0.402309,0.745026,0.546414,0.745026
98,PASTIS_local_T31TFM,5,PASTIS_local_T31TFM,2026-05-09_09-44-07,/home/user/results_local/finetuning_results/PA...,0.552785,0.741629,0.439665,0.741629,0.330758,0.589357,0.402281,0.741629,0.552785,0.741629


In [4]:
metadata_cols = [
    "target_region",
    "val_fold",
    "source_region",
    "seed_or_run",
    "checkpoint_path",
]

metric_cols = [
    col for col in df_local_results.columns
    if col not in metadata_cols
]

# Average over seeds/runs
df_local_seed_avg = (
    df_local_results
    .groupby(["target_region", "val_fold", "source_region"], as_index=False)[metric_cols]
    .mean()
)

# Average over validation folds
df_local_region_avg = (
    df_local_seed_avg
    .groupby(["target_region", "source_region"], as_index=False)[metric_cols]
    .mean()
)

# Final average over all local baselines
df_local_final_avg = (
    df_local_region_avg[metric_cols]
    .mean()
    .to_frame()
    .T
)

df_local_region_avg.to_csv(
    "local_baseline_avg_per_region.csv",
    index=False,
)

df_local_final_avg.to_csv(
    "local_baseline_final_avg_all_regions.csv",
    index=False,
)

df_local_region_avg, df_local_final_avg

(         target_region        source_region  accuracy_macro  accuracy_micro  \
 0  PASTIS_local_T30UXV  PASTIS_local_T30UXV        0.490737        0.704955   
 1  PASTIS_local_T31TFJ  PASTIS_local_T31TFJ        0.344321        0.592736   
 2  PASTIS_local_T31TFM  PASTIS_local_T31TFM        0.561712        0.752315   
 3  PASTIS_local_T32ULU  PASTIS_local_T32ULU        0.624794        0.750247   
 
    f1_macro  f1_micro  jaccard_macro  jaccard_micro  precision_macro  \
 0  0.402805  0.704955       0.310303       0.544401         0.377044   
 1  0.286049  0.592736       0.189576       0.421450         0.265493   
 2  0.458744  0.752315       0.345142       0.603007         0.422001   
 3  0.494660  0.750247       0.369402       0.600350         0.440668   
 
    precision_micro  recall_macro  recall_micro  
 0         0.704955      0.490737      0.704955  
 1         0.592736      0.344321      0.592736  
 2         0.752315      0.561712      0.752315  
 3         0.750247      0.6247